In [52]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from collections import defaultdict
from data_preprocess import PCA, FeatureSelection

In [53]:
class NaiveBayesClassifier:
    def __init__(self, data=None):
        self.edidble_counts = 0
        self.poisonous_counts = 0
        self.edidble_prob = 0.5
        self.poisonous_prob = 0.5
        self.data = data
        self.edidble_features_likelihoods = {}
        self.poisonous_features_likelihoods = {}
        
    def loadData(self, data):
        self.data = data

    def compute_initial_probs(self):
        data = self.data
        nrecords = len(data)
        
        counts = data['class'].value_counts()
        edidble_counts = counts['e']
        poisonous_counts = counts['p']
        
        total = edidble_counts + poisonous_counts
        
        self.edidble_counts = edidble_counts
        self.poisonous_counts = poisonous_counts
        
        self.edidble_prob = edidble_counts / total
        self.poisonous_prob = poisonous_counts / total
    
        
        return self.edidble_prob, self.poisonous_prob
    
    def compute_histograms(self):
        df = self.data
        
        edidble_features_freq = defaultdict(dict)
        poisonous_features_freq = defaultdict(dict)
        total_edible_observations = self.edidble_counts * 22
        total_poisonous_observations = self.poisonous_counts * 22
        
        for _, row in df.iterrows():
            label = row['class']
            
            for feature in df.columns:
                if feature == 'class': continue
                
                value = row[feature]
                if label == 'e':
                    edidble_features_freq[feature][value] = edidble_features_freq[feature].get(value, 0) + 1
                else:
                    poisonous_features_freq[feature][value] = poisonous_features_freq[feature].get(value, 0) + 1
                    
        return edidble_features_freq, poisonous_features_freq, total_edible_observations, total_poisonous_observations
    

    def compute_likelihoods(self): 
        edidble_features_freq, poisonous_features_freq, total_edible_observations, total_poisonous_observations = self.compute_histograms()
        
        edidble_features_likelihoods = defaultdict(dict)
        poisonous_features_likelihoods = defaultdict(dict)
        
        for feature in edidble_features_freq:
            for value, freq in edidble_features_freq[feature].items():
                edidble_features_likelihoods[feature][value] = freq / total_edible_observations
        
        for feature in poisonous_features_freq:
            for value, freq in poisonous_features_freq[feature].items():
                poisonous_features_likelihoods[feature][value] = freq / total_poisonous_observations
                
        self.edidble_features_likelihoods, self.poisonous_features_likelihoods = edidble_features_likelihoods, poisonous_features_likelihoods
                    
        return self.edidble_features_likelihoods, self.poisonous_features_likelihoods
    
    def fit(self):
        self.compute_initial_probs()
        return self.compute_likelihoods()
    
    def predict(self, row: pd.Series):
        edidble_pred_score = np.log(self.edidble_prob)
        poisonous_pred_score = np.log(self.poisonous_prob)
        
        edidble_features_likelihoods = self.edidble_features_likelihoods
        poisonous_features_likelihoods = self.poisonous_features_likelihoods
        
        for feature, value in row.items():
            if feature == 'class': continue
            edidble_pred_score += np.log(edidble_features_likelihoods[feature].get(value, 1e-10))
            poisonous_pred_score += np.log(poisonous_features_likelihoods[feature].get(value, 1e-10))
            

        if edidble_pred_score > poisonous_pred_score:
            return 'e'
        else:
            return 'p'
        
        
    def test(self, test_data: pd.DataFrame):
        for _, row in test_data.iterrows():
            label = row['class']
            pred = self.predict(row=row)
            print(f'{_}: {"correct" if label == pred else "wrong"}')
            

In [ ]:
df = pd.read_csv('data/mushrooms.csv')
train_df, test_df = train_test_split(df, test_size=0.2)
nb = NaiveBayesClassifier(data=train_df)
nb.fit()
nb.test(test_data=test_df)